# Fase CRISP-DM: Limpieza y Feature Engineering Multidominio
**Plataforma**: AgroData Intelligence Platform (AgroStatsApp)  
**Estándares**: FAO-56 Irrigation & Drainage, Time-Series Feature Store, SWEBOK  
**Propósito**: Depurar tipos de datos y anomalías de rango, construir rezagos temporales (lags), ventanas móviles de volatilidad, balances hídricos FAO-56, variables estacionales cíclicas e integrar la matriz maestra de características para modelado predictivo.

In [ ]:
# 1. Configuración de Entorno Resiliente (Google Colab / VS Code / Jupyter Local)
import os
import sys
import subprocess
from pathlib import Path

def setup_environment():
    # A. Detección y preparación automática para Google Colab
    if 'google.colab' in sys.modules or Path('/content').exists():
        print('[INFO] Entorno detectado: Google Colab.')
        repo_dir = Path('/content/Statsfirm')
        if not repo_dir.exists():
            print('[INFO] Clonando repositorio oficial Statsfirm en Colab...')
            subprocess.run(['git', 'clone', 'https://github.com/adansanchezc1-spec/Statsfirm.git', '/content/Statsfirm'], check=True)
        else:
            print('[INFO] Actualizando repositorio en Colab...')
            subprocess.run(['git', '-C', '/content/Statsfirm', 'pull'], check=False)
        
        app_dir = repo_dir / 'AgroStats AndTech' / 'AgroStatsApp'
        if app_dir.exists():
            os.chdir(str(app_dir))
            src_dir = app_dir / 'src'
            if str(src_dir) not in sys.path:
                sys.path.insert(0, str(src_dir))
            print(f'[OK] Directorio de trabajo establecido en: {app_dir}')
            print(f'[OK] Carpeta src agregada a sys.path: {src_dir}')
            return

    # B. Detección dinámica en Entorno Local (Windows / Linux / WSL / VS Code)
    candidates = [
        Path.cwd() / 'src',
        Path.cwd() / 'AgroStats AndTech' / 'AgroStatsApp' / 'src',
        Path.cwd().parent / 'src',
        Path.cwd().parent.parent / 'src',
        Path.cwd().parent.parent / 'AgroStats AndTech' / 'AgroStatsApp' / 'src',
        Path(r'c:\Users\ADAN\OneDrive\Documentos\Statsfirm\AgroStats AndTech\AgroStatsApp\src'),
        Path('/mnt/c/Users/ADAN/OneDrive/Documentos/Statsfirm/AgroStats AndTech/AgroStatsApp/src'),
    ]
    
    curr = Path.cwd().resolve()
    for _ in range(6):
        target = curr / 'AgroStats AndTech' / 'AgroStatsApp' / 'src'
        if target.exists() and (target / 'notebook_code').is_dir():
            candidates.insert(0, target)
            break
        curr = curr.parent

    for c in candidates:
        if c.exists() and (c / 'notebook_code').is_dir():
            resolved = str(c.resolve())
            if resolved not in sys.path:
                sys.path.insert(0, resolved)
            print(f'[OK] Módulo src localizado localmente en: {resolved}')
            return

setup_environment()

# Importación del motor de ingeniería de características
from notebook_code import FeatureEngineer, MasterDataManager, load_all_raw_datasets

print('[OK] Módulo FeatureEngineer importado exitosamente.')


In [ ]:
# 2. Carga y Depuración de Datos Crudos
datasets = load_all_raw_datasets()
df_pluvio = datasets['ideam_pluvio']
df_evap = datasets['ideam_evapotranspiracion']

cleaned_pluvio = FeatureEngineer.clean_dataset(
    df_pluvio,
    date_col='fechaobservacion',
    numeric_cols=['valorobservado'],
    clip_negative_cols=['valorobservado']
)
print(f'Registros limpios en pluviometría: {len(cleaned_pluvio)}')


In [ ]:
# 3. Generación de Rezagos Temporales (Lags)
lags_df = FeatureEngineer.create_lag_features(
    cleaned_pluvio,
    target_cols=['valorobservado'],
    lags=[1, 2, 7, 30]
)
display(lags_df[['fechaobservacion', 'valorobservado', 'valorobservado_lag_1', 'valorobservado_lag_7']].head(10))


In [ ]:
# 4. Generación de Estadísticos sobre Ventanas Móviles (Media y Volatilidad)
rolling_df = FeatureEngineer.create_rolling_features(
    lags_df,
    target_cols=['valorobservado'],
    windows=[7, 14, 30]
)
display(rolling_df[['fechaobservacion', 'valorobservado', 'valorobservado_rolling_mean_7', 'valorobservado_rolling_std_7']].head(10))


In [ ]:
# 5. Balance Hídrico Agroclimático FAO-56
if 'valorobservado' in df_evap.columns:
    rolling_df['evapotranspiracion'] = df_evap['valorobservado'].reindex(rolling_df.index).fillna(3.5)
    fao_df = FeatureEngineer.create_fao_agroclimate_features(
        rolling_df,
        precip_col='valorobservado',
        evap_col='evapotranspiracion'
    )
    display(fao_df[['fechaobservacion', 'valorobservado', 'evapotranspiracion', 'balance_hidrico_neto_fao', 'deficit_hidrico_riego_fao']].head(10))
else:
    fao_df = rolling_df


In [ ]:
# 6. Codificación Cíclica de Estacionalidad (Seno y Coseno)
cyclical_df = FeatureEngineer.create_cyclical_calendar_features(fao_df, date_col='fechaobservacion')
display(cyclical_df[['fechaobservacion', 'sin_mes', 'cos_mes', 'sin_dia_ano', 'cos_dia_ano']].head(10))


In [ ]:
# 7. Persistencia de la Matriz Maestra en CRISPDM/data/FEATURES/
feat_path = FeatureEngineer.save_feature_matrix(cyclical_df)
print(f'Matriz de características persistida exitosamente en: {feat_path}')
